## 4_model_training_pipeline

My goal with this module is to take the data gathered in gather_historic_data.py script and use it to train a model to predict gas hourly gas burn

In [1]:
import pandas as pd 
import numpy as np 
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from xgboost import XGBRegressor
from dateutil.relativedelta import relativedelta

In [5]:
historic_data_df = pd.read_csv('../data/processed-data/historic_data_df.csv')
historic_data_df[['datetime', 'gas_day']] = historic_data_df[['datetime', 'gas_day']].apply(pd.to_datetime)
historic_data_df['hourly_gas_burn_MMBtu'] = historic_data_df['hourly_gas_burn_MMBtu'].fillna(0) # not needed anymore

historic_data_df.head()

,datetime,site,year,month,day,hour,hour_end,day_of_week,gas_day,hourly_gas_burn_MMBtu,...,wind_speed_forecast,total_offline_forecast,offline_ng_forecast,offline_coal_forecast,load_actual,net_load_actual,wind_actual,temperature_actual,wind_speed_actual,total_outages
0,2023-12-01 01:00:00,PGS,2023,12,1,1,HE2,4,2023-11-30,2319.289701,...,2.80,14626.7,7794.0,6832.7,27907.0,11886.62,15672.910,14.000000,3.45,14626.7
1,2023-12-01 02:00:00,PGS,2023,12,1,2,HE3,4,2023-11-30,1961.642224,...,2.80,14626.7,7794.0,6832.7,27835.0,12345.12,15262.724,9.333333,4.55,14626.7
2,2023-12-01 03:00:00,PGS,2023,12,1,3,HE4,4,2023-11-30,2284.608734,...,2.15,14389.1,7691.4,6697.7,27970.0,13126.33,14808.898,14.000000,2.85,14389.1
3,2023-12-01 04:00:00,PGS,2023,12,1,4,HE5,4,2023-11-30,1961.642224,...,4.65,14389.1,7691.4,6697.7,28471.0,14576.82,14124.747,15.333333,4.00,14389.1
4,2023-12-01 05:00:00,PGS,2023,12,1,5,HE6,4,2023-11-30,2163.225347,...,4.35,14389.1,7691.4,6697.7,29643.0,16708.62,13585.278,13.333333,5.15,14389.1


In [6]:
# to create multiple folds for CV just shift all time periods back 7 days then retrain and evaluate
holdout_end_date = '2026-08-06'
holdout_start_date = '2026-07-31'


validation_end_date = '2026-07-31'
validation_start_date = '2026-07-24'


training_end_date = '2026-07-24'
training_start_date = '2024-07-24'

In [7]:
holdout_df = historic_data_df.loc[ (historic_data_df['datetime'] >= holdout_start_date) & (historic_data_df['datetime'] < holdout_end_date) ]
validation_df = historic_data_df.loc[ (historic_data_df['datetime'] >= validation_start_date) & (historic_data_df['datetime'] < validation_end_date) ]
training_df = historic_data_df.loc[ (historic_data_df['datetime'] >= training_start_date) & (historic_data_df['datetime'] < training_end_date) ]

In [8]:
print(f'Historic holdout start date: {holdout_df['datetime'].min()}   Historic holdout end date: {holdout_df['datetime'].max()}')
print(f'Historic validation start date: {validation_df['datetime'].min()}   Historic validation end date: {validation_df['datetime'].max()}')
print(f'Historic training start date: {training_df['datetime'].min()}   Historic training end date: {training_df['datetime'].max()}')

Historic holdout start date: 2026-07-31 00:00:00   Historic holdout end date: 2026-08-05 23:00:00
Historic validation start date: 2026-07-24 00:00:00   Historic validation end date: 2026-07-30 23:00:00
Historic training start date: 2024-07-24 00:00:00   Historic training end date: 2026-07-23 23:00:00


### Segmenting data into training, test, and holdout sets

This section of the notebook is to create a process that progressively advances through the historic data for evaluating the accuracy of the forecasting/regression method

In [9]:
anchor_date = '2026-07-31' # this will be the last day for which actual historic data exists. It will be the last day of the holdout set. Other dates for the fitting process will be based off of this date
evaluation_periods = 0 # this will be the number of different historic time periods to be evaluated. It is the number of times the date will be 'set back' in order to evaluate forecast accuracy

holdout_end_date = pd.to_datetime(anchor_date) + relativedelta(hours=23)
holdout_start_date = pd.to_datetime(holdout_end_date) - relativedelta(hours=7*24-1)
#print(f'Holdout start date: {holdout_start_date}')
#print(f'Holdout end date: {holdout_end_date}\n')


validation_end_date = holdout_end_date - relativedelta(days=7)
validation_start_date = holdout_start_date - relativedelta(days=7)
#print(f'Validation start date: {validation_start_date}')
#print(f'Validation end date: {validation_end_date}\n')


training_end_date = validation_start_date - relativedelta(hours=1)
training_start_date = validation_start_date - relativedelta(years=2)
#print(f'Training start date: {training_start_date}')
#print(f'Training end date: {training_end_date}')

In [10]:
features = ['datetime', 'gas_day', 'year', 'month', 'day_of_week', 'day', 'hour', 'site', 'wind_speed_actual', 'temperature_actual']
non_predictive_features = ['datetime', 'gas_day', 'site'] # these are needed to keep track of metadata, but not needed to actually fit the data

training_data_df = historic_data_df.loc[historic_data_df['datetime'].between(training_start_date, training_end_date), :]
X_training = training_data_df.loc[:, features]
y_training = training_data_df.loc[:, ['site', 'hourly_gas_burn_MMBtu']]

validation_data_df = historic_data_df.loc[historic_data_df['datetime'].between(validation_start_date, validation_end_date), :]
X_validation = validation_data_df.loc[:, features]
y_validation = validation_data_df.loc[:, ['site', 'hourly_gas_burn_MMBtu']] 

holdout_data_df = historic_data_df.loc[historic_data_df['datetime'].between(holdout_start_date, holdout_end_date), :]
X_holdout = holdout_data_df.loc[:, features]
y_holdout = holdout_data_df.loc[:, ['site', 'hourly_gas_burn_MMBtu']] 

#print(training_data_df.shape)
#print(X_training['datetime'].min())
#print(X_training['datetime'].max())

#print(validation_data_df.shape)
#print(X_validation['datetime'].min())
#print(X_validation['datetime'].max())

#print(holdout_data_df.shape)
#print(X_holdout['datetime'].min())
#print(X_holdout['datetime'].max())

## Fitting on the Training Data and Evaluating on the Validation Data

First iteration will focus on a single training time period. Eventually, the goal will be to have the process iterate over multiple time periods to get a sense of how accurate the modelling process will be over time. This will likely turn into adding an additional loop that also sets the 'anchor_date' and recalculates and stores the results of the fitting process.

Potential Ways to Improve or streamline:
- single function to select date, number of evaluation periods, input features, model type and hyperparameters?

What does the looping process look like:
- Walkforward Backtest
- need to loop through sites, dates, hyperparameter settings and collect fit/accuracy details for each
- outer loop => refit frequency
    - inner loop 1 => sites (five total)
        - inner loop 2 => hyperparameters
            - inner loop 3 => forecasted days (days 1-7, days 2-8, days 3-9, etc)


Remember to grab the three error metrics for each model fitted, that is for each time iteration
1. training error 
2. best validation error (i.e. fitting metrics for the best model)
3. holdout error
            

In [11]:
sites =  historic_data_df['site'].unique()
print(sites)

<ArrowStringArray>
['PGS', 'LCS', 'GGS', 'DCS', 'CGS']
Length: 5, dtype: str


In [56]:
end_date = pd.to_datetime('2026-07-31')
anchor_dates = [(end_date - relativedelta(weeks=i)).strftime('%Y-%m-%d') for i in range(10)]
anchor_dates

['2026-07-31',
 '2026-07-24',
 '2026-07-17',
 '2026-07-10',
 '2026-07-03',
 '2026-06-26',
 '2026-06-19',
 '2026-06-12',
 '2026-06-05',
 '2026-05-29']

In [ ]:
training_results = []
for anchor_date in anchor_dates: 
    print(f'Running anchor date: {anchor_date}')
    #calculating start and end dates for modelling dataframes
    holdout_end_date = pd.to_datetime(anchor_date) + relativedelta(hours=23)
    holdout_start_date = pd.to_datetime(holdout_end_date) - relativedelta(hours=7*24-1)

    validation_end_date = holdout_end_date - relativedelta(days=7)
    validation_start_date = holdout_start_date - relativedelta(days=7)

    training_end_date = validation_start_date - relativedelta(hours=1)
    training_start_date = validation_start_date - relativedelta(years=2)

    # creating dataframes based on start and end dates
    training_data_df = historic_data_df.loc[historic_data_df['datetime'].between(training_start_date, training_end_date), :]
    X_training = training_data_df.loc[:, features]
    y_training = training_data_df.loc[:, ['site', 'hourly_gas_burn_MMBtu']]

    validation_data_df = historic_data_df.loc[historic_data_df['datetime'].between(validation_start_date, validation_end_date), :]
    X_validation = validation_data_df.loc[:, features]
    y_validation = validation_data_df.loc[:, ['site', 'hourly_gas_burn_MMBtu']] 

    holdout_data_df = historic_data_df.loc[historic_data_df['datetime'].between(holdout_start_date, holdout_end_date), :]
    X_holdout = holdout_data_df.loc[:, features]
    y_holdout = holdout_data_df.loc[:, ['site', 'hourly_gas_burn_MMBtu']]

    
    for site in sites:

        #print(f"\nTraining model for: {site}")

        X_train_site = X_training[X_training["site"] == site].drop(non_predictive_features, axis=1)
        y_train_site = y_training[y_training['site']== site]['hourly_gas_burn_MMBtu']

        X_validation_site = X_validation[X_validation["site"] == site].drop(non_predictive_features, axis=1)
        y_validation_site = y_validation[y_validation['site']== site]['hourly_gas_burn_MMBtu']

        X_holdout_site = X_holdout[X_holdout["site"] == site].drop(non_predictive_features, axis=1)
        y_holdout_site = y_holdout[y_holdout['site']== site]['hourly_gas_burn_MMBtu']


        ## WHAT OTHER PIPELINE STEPS NEED TO BE COMPLETED HERE...
            # are there any features that need to creatd after the data split occurs
            # need to evaluate the data transformations that are being done to the YES Energy Data => I think there are some aggregations being done that may not be correct
        
        
        # CALEB NOTE: Should we do some type of tuning??
        model = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42)

        model.fit(X_train_site, y_train_site)

        #training error

        training_site_predictions = model.predict(X_train_site)
        validation_site_predictions = model.predict(X_validation_site)
        holdout_site_predictions = model.predict(X_holdout_site)


        training_rmse = round(root_mean_squared_error(y_train_site, training_site_predictions), 4)
        validation_rmse = round(root_mean_squared_error(y_validation_site, validation_site_predictions), 4)
        holdout_rmse = round(root_mean_squared_error(y_holdout_site, holdout_site_predictions), 4)
        
        training_mae = round(mean_absolute_error(y_train_site, training_site_predictions), 4)
        validation_mae = round(mean_absolute_error(y_validation_site, validation_site_predictions), 4)
        holdout_mae = round(mean_absolute_error(y_holdout_site, holdout_site_predictions), 4)


        # need to gather data to predict using gather_future_data. I think the function would be limited by the holdout dates, but tested against the actual
        # I think the holdout RMSE and MAE need to be y_holdout site vs predictions from the gather_future_data if possible. Might be an issue with availability information though...Need to think about this
            # is using the actual values that are currently in PCI for availability a viable option
        # predict future date values
        output_record = {
            "site": site, 
            "anchor_date": anchor_date, 
            "training_start_date": training_start_date, 
            "validation_start_date": validation_start_date, 
            "holdout_start_date": holdout_start_date,
            "training_rmse": training_rmse,
            "validation_rmse": validation_rmse,
            "holdout_rmse": holdout_rmse,
            "training_mae": training_mae,
            "validation_mae": validation_mae,
            "holdout_mae": holdout_mae

        }


        training_results.append(output_record)

training_results = pd.DataFrame(training_results)

    #print(f'Summary for date iteration {anchor_date}')
    #print(f'Training Dates go from {training_start_date} to {training_end_date}')
    #print(f'Validation Dates go from {validation_start_date} to {validation_end_date}')
    #print(f'Holdout Dates go from {holdout_start_date} to {holdout_end_date}')
    #print(output_record)

        #site_hourly_predictions[site] = sites_df

Running anchor date: 2026-07-31
Running anchor date: 2026-07-24
Running anchor date: 2026-07-17
Running anchor date: 2026-07-10
Running anchor date: 2026-07-03
Running anchor date: 2026-06-26
Running anchor date: 2026-06-19
Running anchor date: 2026-06-12
Running anchor date: 2026-06-05
Running anchor date: 2026-05-29


In [64]:
training_results.groupby(by='site').agg({
    'anchor_date': 'count',
    'training_start_date': 'min',
    'validation_start_date': 'min',
    'holdout_start_date': 'min',
    'training_rmse': 'mean',
    'validation_rmse': 'mean',
    'holdout_rmse': 'mean',
    'training_mae': 'mean',
    'validation_mae': 'mean',
    'holdout_mae': 'mean'
})


,anchor_date,training_start_date,validation_start_date,holdout_start_date,training_rmse,validation_rmse,holdout_rmse,training_mae,validation_mae,holdout_mae
site,,,,,,,,,,
CGS,10,2024-05-16,2026-05-16,2026-05-23,143.08235,119.84476,182.07790,86.36203,105.90364,170.37455
DCS,10,2024-05-16,2026-05-16,2026-05-23,716.12876,1971.83462,1882.85911,496.64242,1571.02174,1522.82539
GGS,10,2024-05-16,2026-05-16,2026-05-23,329.95813,959.92549,1188.47150,205.58863,642.32622,776.66787
LCS,10,2024-05-16,2026-05-16,2026-05-23,489.93463,1316.04956,1403.06475,331.16581,1068.36323,1147.52629
PGS,10,2024-05-16,2026-05-16,2026-05-23,1371.68708,5573.63265,6830.48897,801.68757,4634.00196,5831.45446
